# Minimal RAG From Scratch

| Field | Value |
|---|---|
| Stage | Foundations |
| Difficulty | Beginner to intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-24 |

Callout - Key idea:
A minimal RAG system is an inspectable composition of representation, retrieval, context selection, generation, citation, and abstention—not a framework call.

## 30-Second Summary

This notebook builds a small lexical RAG pipeline using only Python's standard library. It compares raw token overlap with TF-IDF cosine retrieval, selects one evidence sentence as the deterministic “generator,” cites its document ID, and abstains when the question lacks enough evidence.

## Why This Matters

Frameworks make assembly faster, but abstractions can hide why a document ranked first or why a system answered without support. Building one transparent baseline gives us a debugging reference for every later embedding, vector-store, reranking, and agentic lesson.

## Scope

| Covers | Does not cover |
|---|---|
| Tokenization, TF-IDF, cosine similarity, top-k retrieval, extractive generation, citation, abstention, retrieval metrics | Semantic embeddings, learned rerankers, LLM generation, chunking large documents |

### Prerequisites

Complete the RAG mental-model notebook first.

## Mental Model

```text
documents -> tokenize -> TF-IDF vectors -> searchable matrix
                                               |
question  -> tokenize -> TF-IDF vector  -> cosine ranking -> top-k
                                                        |
                                      best supported sentence + source ID
```

TF-IDF gives more weight to terms that are frequent in one document but uncommon across the corpus. Cosine similarity compares vector direction, reducing the effect of document length. This is lexical retrieval: it understands shared terms, not meaning or paraphrases.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import math
import re


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')


def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as source:
        return [json.loads(line) for line in source if line.strip()]


REPO_ROOT = find_repo_root()
documents = load_jsonl(REPO_ROOT / 'data/raw/rag_101_corpus.jsonl')
questions = load_jsonl(REPO_ROOT / 'data/evaluation/golden_questions.jsonl')
len(documents), len(questions)

## How It Works

For term $t$ in document $d$:

$$TFIDF(t,d) = (1 + \log(count(t,d))) \times (\log((1+N)/(1+df(t))) + 1)$$

The smoothed inverse document frequency avoids division by zero. We L2-normalize each vector, so its dot product with another normalized vector is cosine similarity. Higher scores mean stronger lexical similarity.

The implementation deliberately exposes the vocabulary, IDF values, vectors, and scores. Production libraries optimize these operations, but the responsibilities remain the same.

## Baseline

The weakest reasonable baseline counts unique query tokens appearing in each document. It ignores term rarity and document length. That makes a generic document able to tie with a more specific one.

In [ ]:
STOP_WORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'how',
    'in', 'is', 'it', 'of', 'on', 'or', 'the', 'to', 'what', 'when', 'which', 'with'
}


def tokenize(text: str) -> list[str]:
    return [
        token for token in re.findall(r'[a-z0-9]+', text.lower())
        if len(token) > 1 and token not in STOP_WORDS
    ]


def overlap_search(query: str, k: int = 3) -> list[dict]:
    query_terms = set(tokenize(query))
    scored = [
        {**document, 'score': len(query_terms.intersection(tokenize(document['content'])))}
        for document in documents
    ]
    return sorted(scored, key=lambda item: (-item['score'], item['id']))[:k]


[(item['id'], item['score']) for item in overlap_search(questions[3]['question'])]

## Technique Implementation

We now fit TF-IDF on the document collection, transform documents and questions into the same vocabulary, normalize the vectors, and rank by cosine similarity. Fitting belongs to the offline path; transforming a question and searching belong to the online path.

In [ ]:
class TfidfVectorizer:
    def fit(self, texts: list[str]) -> 'TfidfVectorizer':
        document_frequency = Counter()
        for text in texts:
            document_frequency.update(set(tokenize(text)))
        terms = sorted(document_frequency)
        self.vocabulary = {term: index for index, term in enumerate(terms)}
        self.idf = [
            math.log((1 + len(texts)) / (1 + document_frequency[term])) + 1
            for term in terms
        ]
        return self

    def transform_one(self, text: str) -> list[float]:
        counts = Counter(tokenize(text))
        vector = [0.0] * len(self.vocabulary)
        for term, count in counts.items():
            if term in self.vocabulary:
                index = self.vocabulary[term]
                vector[index] = (1 + math.log(count)) * self.idf[index]
        norm = math.sqrt(sum(value * value for value in vector))
        return [value / norm for value in vector] if norm else vector


def cosine(left: list[float], right: list[float]) -> float:
    return sum(a * b for a, b in zip(left, right, strict=True))


vectorizer = TfidfVectorizer().fit([document['content'] for document in documents])
document_vectors = [vectorizer.transform_one(document['content']) for document in documents]


def tfidf_search(query: str, k: int = 3) -> list[dict]:
    query_vector = vectorizer.transform_one(query)
    scored = [
        {**document, 'score': cosine(query_vector, vector)}
        for document, vector in zip(documents, document_vectors, strict=True)
    ]
    return sorted(scored, key=lambda item: (-item['score'], item['id']))[:k]


[(item['id'], round(item['score'], 3)) for item in tfidf_search(questions[3]['question'])]

## Controlled Experiment

Both retrievers use the same tokenizer, documents, questions, and `k=1`. The only change is scoring: raw unique-token overlap versus normalized TF-IDF cosine similarity.

Hit rate@1 asks whether the expected source is first. MRR gives reciprocal credit for its first relevant rank; with `k=1` the values match, but the function remains useful when we inspect deeper rankings.

In [ ]:
def retrieval_metrics(search, k: int = 1) -> dict[str, float]:
    answerable = [question for question in questions if question['relevant_doc_ids']]
    hits = []
    reciprocal_ranks = []
    for question in answerable:
        ranked_ids = [item['id'] for item in search(question['question'], k)]
        relevant = set(question['relevant_doc_ids'])
        hits.append(float(bool(set(ranked_ids).intersection(relevant))))
        first_rank = next((i for i, doc_id in enumerate(ranked_ids, 1) if doc_id in relevant), None)
        reciprocal_ranks.append(1 / first_rank if first_rank else 0.0)
    return {
        f'hit_rate@{k}': sum(hits) / len(hits),
        'mrr': sum(reciprocal_ranks) / len(reciprocal_ranks),
    }


experiment_results = {
    'raw_overlap': retrieval_metrics(overlap_search),
    'tfidf_cosine': retrieval_metrics(tfidf_search),
}
experiment_results

## Evaluation

On this deliberately small corpus, TF-IDF places the expected document first for all seven answerable questions. Raw overlap confuses the Priority 1 support question with the backup policy because both contain common incident terms. This proves behavior only on this golden set; it does not prove semantic understanding or production readiness.

| Signal | Raw overlap | TF-IDF cosine | Meaning |
|---|---:|---:|---|
| Hit rate@1 | 0.857 | 1.000 | Expected source ranked first |
| MRR at depth 1 | 0.857 | 1.000 | First relevant rank at this cutoff |

In [ ]:
assert experiment_results['raw_overlap']['hit_rate@1'] < 1.0
assert experiment_results['tfidf_cosine']['hit_rate@1'] == 1.0
assert tfidf_search(questions[3]['question'], 1)[0]['id'] == 'northstar-incidents'
print('Retrieval checks passed.')

## Decision Guide

| Situation | Choose | Reason | Trade-off |
|---|---|---|---|
| Exact product names, error codes, legal phrases | Lexical retrieval | Literal terms are strong signals | Misses paraphrases |
| Natural-language paraphrases and conceptual similarity | Dense embeddings | Represents semantic similarity | Model cost and domain mismatch |
| Mixed technical and conversational queries | Hybrid retrieval | Combines exact and semantic strengths | Fusion and tuning complexity |
| Tiny transparent learning baseline | This implementation | Every score is inspectable | Not optimized for scale |

Do not use this implementation as a large production index. Its purpose is understanding and regression baselining.

## Failure Modes and Debugging

| Symptom | Likely cause | How to verify | Fix |
|---|---|---|---|
| Paraphrase retrieves nothing | No shared vocabulary | Inspect query/document tokens | Add dense or hybrid retrieval |
| Generic document ranks too high | Common terms dominate | Inspect term document frequencies | Use IDF, filters, or reranking |
| Long document dilutes the match | Representation too broad | Compare chunk-level and document-level scores | Chunk at meaningful boundaries |
| Unsupported question still gets an answer | No evidence threshold | Add unanswerable golden questions | Calibrate abstention and validate support |
| Metric looks perfect | Dataset is too easy or leaked | Inspect query variety and labels | Expand and independently review the golden set |

## Production Notes

### Observability
Log ranked IDs and scores, not full sensitive documents. Track score distributions and zero-result rates across query segments.

### Safety and Guardrails
Apply authorization filters before scoring. A highly relevant unauthorized document must never enter the candidate set.

### Latency and Cost
This code scans every document for every query. Production sparse indexes use inverted indexes so work scales with matching terms rather than the entire corpus.

## Practice

Add one paraphrased golden question that contains none of the important words in its relevant document. Predict the lexical result, run it, then record why dense retrieval may help.

## Recall

Toggle - Recall: What does IDF contribute?
It gives uncommon corpus terms more influence than terms appearing in many documents.

Toggle - Recall: Why normalize before cosine comparison?
It compares direction rather than letting vector magnitude or document length dominate.

Toggle - Recall: What can lexical retrieval not understand?
Meaning expressed with different vocabulary unless expansion or another representation bridges it.

Toggle - Recall: Why keep an unanswerable question?
It tests whether the system abstains instead of converting weak similarity into an unsupported answer.

## Sources

- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [scikit-learn: TF-IDF term weighting](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)

The implementation is intentionally educational and uses repository-owned fictional data.

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-24 | Complete | High | Compare with BM25 and dense retrieval in later lessons |